# 03b — Fine-tune DziriBERT End-to-End
**Reddit Opinion & Community Research — refines notebook 03's frozen-embedding labels**



In [ ]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg, imp in [("pandas", None), ("pyarrow", None), ("numpy", None),
                 ("scikit-learn", "sklearn"), ("torch", None), ("transformers", None)]:
    ensure(pkg, imp)

print("Packages OK.")


Packages OK.


In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/PhD/reddit_opinion_research"
else:
    BASE_DIR = os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")

CONTRACT_PATH = os.path.join(BASE_DIR, "processed_v2", "raw_enrichment_contract.json")
with open(CONTRACT_PATH, "r", encoding="utf-8") as f:
    contract = json.load(f)
PROCESSED_V2_DIR = contract["processed_v2_dir"]
GOLD_PATH = os.path.join(BASE_DIR, "annotation", "gold_labels_consensus.parquet")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected -- switch the Colab/Kaggle runtime to GPU before continuing, this will be very slow otherwise.")


Mounted at /content/drive
Using device: cuda


## Load gold data and the full corpus

In [ ]:
text_corpus = pd.read_parquet(contract["paths"]["text_corpus"])
gold = pd.read_parquet(GOLD_PATH)
gold["emotion"] = gold["emotion"].replace({"Fear": "Sadness"})  # same merge as notebook 03

gold_rows = text_corpus.merge(gold[["doc_id", "sentiment", "emotion"]], on="doc_id", how="inner")
print(f"gold_rows: {len(gold_rows):,} documents matched")

MODEL_NAME = "alger-ia/dziribert"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LENGTH = 128
enc = tokenizer(gold_rows["text"].tolist(), padding="max_length", truncation=True,
                 max_length=MAX_LENGTH, return_tensors="pt")
gold_input_ids = enc["input_ids"]
gold_attention_mask = enc["attention_mask"]
print("Tokenized gold set:", gold_input_ids.shape)


gold_rows: 916 documents matched


config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/446k [00:00<?, ?B/s]

Tokenized gold set: torch.Size([916, 128])


## Training function 

In [ ]:
def train_classifier(model_name, num_labels, train_ids, train_mask, train_labels,
                      val_ids, val_mask, val_labels, device,
                      max_epochs=15, patience=3, batch_size=16, lr=2e-5):
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)

    class_counts = np.bincount(train_labels, minlength=num_labels)
    class_weights = torch.tensor(
        len(train_labels) / (num_labels * np.maximum(class_counts, 1)), dtype=torch.float32
    ).to(device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    train_ds = torch.utils.data.TensorDataset(train_ids, train_mask, torch.tensor(train_labels))
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    best_val_f1, best_state, patience_ctr = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for batch_ids, batch_mask, batch_labels in train_loader:
            batch_ids, batch_mask, batch_labels = batch_ids.to(device), batch_mask.to(device), batch_labels.to(device)
            optimizer.zero_grad()
            out = model(input_ids=batch_ids, attention_mask=batch_mask)
            loss = loss_fn(out.logits, batch_labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_out = model(input_ids=val_ids.to(device), attention_mask=val_mask.to(device))
            val_pred = val_out.logits.argmax(dim=-1).cpu().numpy()
        val_f1 = f1_score(val_labels, val_pred, average="macro", zero_division=0)

        if val_f1 > best_val_f1:
            best_val_f1, patience_ctr = val_f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_f1


## Nested 5-fold CV, per task

In [ ]:
def run_nested_cv(task_name, y_raw, n_outer=5):
    label_encoder = LabelEncoder().fit(y_raw)
    y_encoded = label_encoder.transform(y_raw)
    num_labels = len(label_encoder.classes_)

    outer_skf = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(y_encoded), dtype=int) - 1

    for fold_i, (train_idx, test_idx) in enumerate(outer_skf.split(gold_input_ids, y_encoded)):
        tr_idx, val_idx = train_test_split(
            train_idx, test_size=0.15, stratify=y_encoded[train_idx], random_state=0
        )
        model, best_val_f1 = train_classifier(
            MODEL_NAME, num_labels,
            gold_input_ids[tr_idx], gold_attention_mask[tr_idx], y_encoded[tr_idx],
            gold_input_ids[val_idx], gold_attention_mask[val_idx], y_encoded[val_idx],
            DEVICE,
        )
        model.eval()
        with torch.no_grad():
            test_out = model(
                input_ids=gold_input_ids[test_idx].to(DEVICE),
                attention_mask=gold_attention_mask[test_idx].to(DEVICE),
            )
            test_pred = test_out.logits.argmax(dim=-1).cpu().numpy()
        oof_preds[test_idx] = test_pred
        test_f1 = f1_score(y_encoded[test_idx], test_pred, average="macro", zero_division=0)
        print(f"[{task_name}] Fold {fold_i}: inner-val best macro_f1={best_val_f1:.3f}, "
              f"outer-test macro_f1={test_f1:.3f}")

        del model
        torch.cuda.empty_cache()

    print(f"\n[{task_name}] Overall out-of-fold report:")
    print(classification_report(y_encoded, oof_preds, target_names=label_encoder.classes_, zero_division=0))
    return label_encoder, y_encoded, oof_preds

sentiment_encoder, sentiment_y, sentiment_oof = run_nested_cv("sentiment", gold_rows["sentiment"].values)


model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[sentiment] Fold 0: inner-val best macro_f1=0.694, outer-test macro_f1=0.625


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

... (7 outputs left out)

[sentiment] Fold 3: inner-val best macro_f1=0.723, outer-test macro_f1=0.645


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[sentiment] Fold 4: inner-val best macro_f1=0.641, outer-test macro_f1=0.647

[sentiment] Overall out-of-fold report:
              precision    recall  f1-score   support

    Negative       0.70      0.70      0.70       374
     Neutral       0.61      0.63      0.62       332
    Positive       0.62      0.60      0.61       210

    accuracy                           0.65       916
   macro avg       0.64      0.64      0.64       916
weighted avg       0.65      0.65      0.65       916



In [ ]:
emotion_encoder, emotion_y, emotion_oof = run_nested_cv("emotion", gold_rows["emotion"].values)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[emotion] Fold 0: inner-val best macro_f1=0.482, outer-test macro_f1=0.510


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[emotion] Fold 1: inner-val best macro_f1=0.632, outer-test macro_f1=0.557


... (5 outputs left out)

[emotion] Fold 3: inner-val best macro_f1=0.431, outer-test macro_f1=0.414


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[emotion] Fold 4: inner-val best macro_f1=0.452, outer-test macro_f1=0.625

[emotion] Overall out-of-fold report:
              precision    recall  f1-score   support

       Anger       0.66      0.65      0.66       250
         Joy       0.62      0.63      0.62       179
     Neutral       0.66      0.68      0.67       360
     Sadness       0.43      0.41      0.42        86
    Surprise       0.29      0.24      0.26        41

    accuracy                           0.62       916
   macro avg       0.53      0.52      0.53       916
weighted avg       0.61      0.62      0.61       916



## Final deployed models

In [ ]:
def train_final_model(task_name, y_encoded, num_labels):
    tr_idx, val_idx = train_test_split(
        np.arange(len(y_encoded)), test_size=0.10, stratify=y_encoded, random_state=0
    )
    model, best_val_f1 = train_classifier(
        MODEL_NAME, num_labels,
        gold_input_ids[tr_idx], gold_attention_mask[tr_idx], y_encoded[tr_idx],
        gold_input_ids[val_idx], gold_attention_mask[val_idx], y_encoded[val_idx],
        DEVICE,
    )
    print(f"[{task_name}] Final model: inner-val best macro_f1={best_val_f1:.3f} "
          f"(trained on {len(tr_idx)}/{len(y_encoded)} gold docs)")
    return model

sentiment_model = train_final_model("sentiment", sentiment_y, len(sentiment_encoder.classes_))


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[sentiment] Final model: inner-val best macro_f1=0.756 (trained on 824/916 gold docs)


In [ ]:
emotion_model = train_final_model("emotion", emotion_y, len(emotion_encoder.classes_))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[emotion] Final model: inner-val best macro_f1=0.497 (trained on 824/916 gold docs)


## Full-corpus inference 

In [ ]:
@torch.no_grad()
def predict_in_chunks(model, texts, task_name, chunks_dir, chunk_size=20000, batch_size=64, max_length=128):
    os.makedirs(chunks_dir, exist_ok=True)
    n_total = len(texts)
    n_chunks = (n_total + chunk_size - 1) // chunk_size
    model.eval()

    for chunk_idx in range(n_chunks):
        chunk_path = os.path.join(chunks_dir, f"{task_name}_chunk_{chunk_idx:04d}.npy")
        if os.path.isfile(chunk_path):
            print(f"[{task_name}] Chunk {chunk_idx + 1}/{n_chunks} already done, skipping.")
            continue
        start, end = chunk_idx * chunk_size, min((chunk_idx + 1) * chunk_size, n_total)
        chunk_texts = texts[start:end]
        preds = []
        for i in range(0, len(chunk_texts), batch_size):
            batch = [t if isinstance(t, str) and t else " " for t in chunk_texts[i:i + batch_size]]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            out = model(**enc)
            preds.append(out.logits.argmax(dim=-1).cpu().numpy())
        chunk_preds = np.concatenate(preds)
        np.save(chunk_path, chunk_preds)
        print(f"[{task_name}] Chunk {chunk_idx + 1}/{n_chunks} done ({end:,}/{n_total:,} docs) -> {chunk_path}")

    return np.concatenate([np.load(os.path.join(chunks_dir, f"{task_name}_chunk_{i:04d}.npy")) for i in range(n_chunks)])

chunks_dir = os.path.join(PROCESSED_V2_DIR, "finetuned_prediction_chunks")
texts = text_corpus["text"].tolist()

sentiment_pred_idx = predict_in_chunks(sentiment_model, texts, "sentiment", chunks_dir)


[sentiment] Chunk 1/11 done (20,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/sentiment_chunk_0000.npy
[sentiment] Chunk 2/11 done (40,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/sentiment_chunk_0001.npy
[sentiment] Chunk 3/11 done (60,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/sentiment_chunk_0002.npy
[sentiment] Chunk 4/11 done (80,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/sentiment_chunk_0003.npy
[sentiment] Chunk 5/11 done (100,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/sentiment_chunk_0004.npy
[sentiment] Chunk 6/11 done (120,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/sen

In [ ]:
emotion_pred_idx = predict_in_chunks(emotion_model, texts, "emotion", chunks_dir)

[emotion] Chunk 1/11 done (20,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/emotion_chunk_0000.npy
[emotion] Chunk 2/11 done (40,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/emotion_chunk_0001.npy
[emotion] Chunk 3/11 done (60,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/emotion_chunk_0002.npy
[emotion] Chunk 4/11 done (80,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/emotion_chunk_0003.npy
[emotion] Chunk 5/11 done (100,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/emotion_chunk_0004.npy
[emotion] Chunk 6/11 done (120,000/210,994 docs) -> /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetuned_prediction_chunks/emotion_chunk_0005.npy
[e

## Apply predictions to the full corpus

In [ ]:
text_corpus["sentiment_label"] = sentiment_encoder.inverse_transform(sentiment_pred_idx)
text_corpus["emotion_label"] = emotion_encoder.inverse_transform(emotion_pred_idx)

# gold rows keep their human/LLM-consensus label, not the model's own prediction on itself
gold_doc_ids = set(gold_rows["doc_id"])
is_gold = text_corpus["doc_id"].isin(gold_doc_ids)
gold_lookup_sentiment = gold_rows.set_index("doc_id")["sentiment"]
gold_lookup_emotion = gold_rows.set_index("doc_id")["emotion"]
text_corpus.loc[is_gold, "sentiment_label"] = text_corpus.loc[is_gold, "doc_id"].map(gold_lookup_sentiment)
text_corpus.loc[is_gold, "emotion_label"] = text_corpus.loc[is_gold, "doc_id"].map(gold_lookup_emotion)

print("Full-corpus predicted sentiment distribution:")
print(text_corpus["sentiment_label"].value_counts(normalize=True).mul(100).round(1))
print("\nFull-corpus predicted emotion distribution:")
print(text_corpus["emotion_label"].value_counts(normalize=True).mul(100).round(1))

labeled_v2_path = os.path.join(PROCESSED_V2_DIR, "text_corpus_labeled_finetuned.parquet")
text_corpus.to_parquet(labeled_v2_path, engine="pyarrow", index=False)
print("\nSaved:", labeled_v2_path)
print("(notebook 03's frozen-embedding version, text_corpus_labeled.parquet, is left untouched for an ablation comparison)")


Full-corpus predicted sentiment distribution:
sentiment_label
Negative    41.3
Neutral     40.9
Positive    17.9
Name: proportion, dtype: float64

Full-corpus predicted emotion distribution:
emotion_label
Neutral     32.2
Anger       24.3
Joy         19.7
Sadness     17.7
Surprise     6.1
Name: proportion, dtype: float64

Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/text_corpus_labeled_finetuned.parquet
(notebook 03's frozen-embedding version, text_corpus_labeled.parquet, is left untouched for an ablation comparison)


## Write the fine-tuning contract

In [ ]:
def cv_report_dict(y_encoded, oof_preds, label_encoder):
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score
    precision, recall, f1, support = precision_recall_fscore_support(y_encoded, oof_preds, zero_division=0)
    per_class = {
        label_encoder.classes_[i]: {"precision": float(precision[i]), "recall": float(recall[i]),
                                     "f1": float(f1[i]), "support": int(support[i])}
        for i in range(len(label_encoder.classes_))
    }
    return {
        "accuracy": float(accuracy_score(y_encoded, oof_preds)),
        "macro_f1": float(f1_score(y_encoded, oof_preds, average="macro", zero_division=0)),
        "per_class": per_class,
    }

finetune_contract = {
    "model": MODEL_NAME,
    "paths": {"text_corpus_labeled_finetuned": labeled_v2_path},
    "nested_cv_results": {
        "sentiment": cv_report_dict(sentiment_y, sentiment_oof, sentiment_encoder),
        "emotion": cv_report_dict(emotion_y, emotion_oof, emotion_encoder),
    },
}

finetune_contract_path = os.path.join(PROCESSED_V2_DIR, "finetune_contract.json")
with open(finetune_contract_path, "w", encoding="utf-8") as f:
    json.dump(finetune_contract, f, indent=2, ensure_ascii=False, default=str)

print("Saved:", finetune_contract_path)
print(json.dumps(finetune_contract["nested_cv_results"], indent=2, ensure_ascii=False))


Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/finetune_contract.json
{
  "sentiment": {
    "accuracy": 0.648471615720524,
    "macro_f1": 0.6411561635295554,
    "per_class": {
      "Negative": {
        "precision": 0.6970509383378016,
        "recall": 0.6951871657754011,
        "f1": 0.6961178045515395,
        "support": 374
      },
      "Neutral": {
        "precision": 0.6147058823529412,
        "recall": 0.6295180722891566,
        "f1": 0.6220238095238095,
        "support": 332
      },
      "Positive": {
        "precision": 0.6157635467980296,
        "recall": 0.5952380952380952,
        "f1": 0.6053268765133172,
        "support": 210
      }
    }
  },
  "emotion": {
    "accuracy": 0.6168122270742358,
    "macro_f1": 0.5257976718413125,
    "per_class": {
      "Anger": {
        "precision": 0.6639344262295082,
        "recall": 0.648,
        "f1": 0.6558704453441295,
        "support": 250
      },
      "Joy": {
        "precision": 0.